In [1]:
#------------------------------------------------ Begin_Librairie ----------------------------------------
import pandas as pd
from time import sleep
import datetime
import re
import os
import requests
import pdfplumber
!pip install -U "googletrans==4.0.2"
from googletrans import Translator
import urllib3
urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)


[notice] A new release of pip is available: 25.3 -> 26.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
#------------------------------------------------ Begin_fileName ----------------------------------------
regulatorName = 'YE CBYE'
print(f"Running {regulatorName} Web Scraping Tool v.1.0")

now = datetime.datetime.now()
filename = '{} SQL Ready {}.xlsx'.format(regulatorName, str(now).replace(':', '.')[:-7])

scriptfolder = f"C:\\Users\\wuj1\\OneDrive - Moody's\\Desktop\\Regulator\\{regulatorName}"
os.chdir(scriptfolder)

tempfolder = os.path.join(scriptfolder, 'tempfolder')
if os.path.exists(tempfolder):
    for rem in os.listdir(tempfolder):
        os.remove(os.path.join(tempfolder, rem))
else:
    os.mkdir(tempfolder)

processdate = now.strftime('%Y-%m-%d')

Running YE CBYE Web Scraping Tool v.1.0


In [3]:
#------------------------------------------------ Begin_Function ----------------------------------------
def bourange_same_length_array(sqldict):
    maxlen = len(sqldict['ListProcessDate'])
    for key in sqldict:
        if len(sqldict[key]) != maxlen:
            sqldict[key] = sqldict[key] + [''] * (maxlen - len(sqldict[key]))
    return sqldict


# ---- regulator-translate skill: reusable helper ----
DEFAULT_SRC = 'ar'
DEFAULT_DEST = 'en'
_translator = Translator()
_translation_cache = {}
_MISSING_TOKENS = {'', '-', '—', 'N/A', 'NA', 'None', 'null', '无', '無'}

def is_missing(value) -> bool:
    if value is None:
        return True
    s = str(value).strip()
    return s == '' or s in _MISSING_TOKENS

def _provider_translate(text: str, src: str, dest: str) -> str:
    return _translator.translate(text, src=src, dest=dest).text

def translate_text(text, src: str = DEFAULT_SRC, dest: str = DEFAULT_DEST,
                   max_retries: int = 3, base_delay: float = 2.0) -> str:
    if is_missing(text):
        return ''
    key = (str(text), src, dest)
    if key in _translation_cache:
        return _translation_cache[key]
    cleaned = str(text).replace('\n', ' ').strip()
    last_err = None
    for attempt in range(1, max_retries + 1):
        try:
            out = _provider_translate(cleaned, src, dest)
            if out:
                _translation_cache[key] = out
                return out
        except Exception as e:
            last_err = e
            print(f"[translate] attempt {attempt}/{max_retries} failed: {e}")
            sleep(base_delay * attempt)
    print(f"[translate] giving up, using original. last error: {last_err}")
    _translation_cache[key] = cleaned
    return cleaned


def download_pdf(url: str, dest_folder: str) -> str:
    """Download a PDF directly into dest_folder; return the local path."""
    fname = url.rstrip('/').split('/')[-1]
    if not fname.lower().endswith('.pdf'):
        fname = fname + '.pdf'
    local = os.path.join(dest_folder, fname)
    headers = {'User-Agent': 'Mozilla/5.0'}
    r = requests.get(url, headers=headers, verify=False, timeout=60)
    r.raise_for_status()
    with open(local, 'wb') as f:
        f.write(r.content)
    print(f"[download] {url} -> {local} ({len(r.content)} bytes)")
    return local


def clear_tempfolder(folder: str):
    folder_abs = os.path.abspath(folder)
    if not folder_abs or folder_abs in ('/', '\\') or not folder_abs.lower().endswith('tempfolder'):
        raise RuntimeError(f'Refusing to clear suspicious path: {folder_abs}')
    for rem in os.listdir(folder_abs):
        os.remove(os.path.join(folder_abs, rem))


_URL_RE = re.compile(r'(?:https?://|www\.)[\w.\-/]+', re.IGNORECASE)

def split_address_website(text: str):
    """Split a free-text cell into (address_without_url, website_or_empty)."""
    if not text:
        return '', ''
    s = str(text).replace('\n', ' ').strip()
    m = _URL_RE.search(s)
    if not m:
        return s, ''
    website = m.group(0)
    address = (s[:m.start()] + s[m.end():]).strip(' ,;-')
    return address, website

In [15]:
#------------------------------------------------ Begin_Variable ----------------------------------------
regdict = {
    'YE CBYE 1': 'https://english.cby-ye.com/pages/14',
    'YE CBYE 2': 'https://english.cby-ye.com/pages/15',
}

Typology = {
    'YE CBYE 1': 'Licensed banks in the Republic of Yemen',
    'YE CBYE 2': 'Licensed exchange companies in the Republic of Yemen',
}

# Direct PDF links per list. List 2 has two Arabic PDFs that feed the same ListCode.
PdfLinks = {
    'YE CBYE 1': ['https://english.cby-ye.com/files/69dfc71a51c14.pdf'],
    'YE CBYE 2': [
        'https://english.cby-ye.com/files/615d6aedbf960.pdf',  # شركات الصرافة
        'https://english.cby-ye.com/files/615d6b5e11ba1.pdf',  # منشئات الصرافة
    ],
}

sqldict = {'bvdid': [], 'priority': [], 'ListLabel': [], 'Typology': [], 'EntryType': [], 'Name': [], 'InternalID_1': [], 'InternalID_1_type': [], 'InternalID_2': [],
           'InternalID_2_type': [], 'InternalID_3': [], 'InternalID_3_type': [], 'CoType': [], 'License_Type': [], 'Address_1': [], 'Address_2': [], 'City': [],
           'Zip': [], 'Cntry': [], 'Phone': [], 'Fax': [], 'Website': [], 'Email': [], 'RegulationType': [], 'RegulationTypeCode': [], 'RegulationDate': [], 'CancellationDate': [],
           'RegCtry': [], 'RegCode': [], 'ListCode': [], 'ListLanguage': [], 'ListValidityDate': [], 'ListName': [], 'ListProcessDate': [], 'LEI Code': [], 'BIC SWIFT Code': [], 'Name - Mother Company': [],
           'Address_1 - Mother company': [], 'Address_2 -  Mother company': [], 'City - Mother company': [], 'Zip - Mother company': [], 'Cntry - Mother company': [],
           'Phone - Mother company': [], 'Check': []}

## List 1 — Licensed banks in the Republic of Yemen (English PDF)

In [5]:
reg = 'YE CBYE 1'
print(f'Working with list {reg}')
clear_tempfolder(tempfolder)

pdf_path = download_pdf(PdfLinks[reg][0], tempfolder)

rows = []
with pdfplumber.open(pdf_path) as pdf:
    for page in pdf.pages:
        tbl = page.extract_table()
        if tbl:
            for r in tbl:
                rows.append(r)
        else:
            # Fallback: split lines into pseudo-rows for pages where no table is detected.
            txt = page.extract_text() or ''
            for line in txt.splitlines():
                if line.strip():
                    rows.append([line.strip()])

print(f'Raw rows extracted: {len(rows)}')
for sample in rows[:5]:
    print(sample)

Working with list YE CBYE 1
[download] https://english.cby-ye.com/files/69dfc71a51c14.pdf -> C:\Users\wuj1\OneDrive - Moody's\Desktop\Regulator\YE CBYE\tempfolder\69dfc71a51c14.pdf (160232 bytes)
Raw rows extracted: 29
['ةينميلا ةيروهملجاب ةصخرلما كونبلاب فشك', None, None, None, None, None]
["Bank's Adress", 'كنبلا ناونع', 'ينوتركللاا عقولما', "Bank's Name", 'كنبلا مسا', 'م']
['Arwa Street, Crater, Aden - Yemen', 'نميلا -ندع-ىورأ عراش', '_', 'Yemen Bank for Reconstruction & Development', 'يرمعتلاو ءاشنلال نيميلا كنبلا', '1']
['Arwa Street, Crater, Aden - Yemen', 'نميلا -ندع-ىورأ عراش', 'www.nbyemen.com', 'National Bank of Yemen', 'نيميلا يلهلأا كنبلا', '2']
["Al-Zubairy Street, Sana'a - Yemen", 'نميلا - ءاعنص - ييربزلا عراش', 'www.arabbank.com', 'Arab Bank PLC - Yemen', 'يبرعلا كنبلا', '3']


In [6]:
# Build a DataFrame with Name / Address / Website. The PDF table is expected to have
# a header row containing 'Bank' / 'Name' / 'Address' / 'Website'; detect it dynamically.
header_idx = None
for i, r in enumerate(rows):
    joined = ' '.join([str(c or '') for c in r]).lower()
    if 'bank' in joined and ('address' in joined or 'website' in joined or 'name' in joined):
        header_idx = i
        break

if header_idx is not None:
    header = [str(c or '').strip() for c in rows[header_idx]]
    body = [r for r in rows[header_idx + 1:] if r and any(str(c or '').strip() for c in r)]
    # Pad / trim every row to header length
    width = len(header)
    body = [list(r) + [''] * (width - len(r)) if len(r) < width else list(r)[:width] for r in body]
    df1 = pd.DataFrame(body, columns=header)
else:
    # Fallback: treat each row's first cell as the bank name only.
    df1 = pd.DataFrame({'Name': [str(r[0]).strip() for r in rows if r and str(r[0]).strip()]})

print(df1.shape)
df1.head(10)

(26, 6)


,Bank's Adress,كنبلا ناونع,ينوتركللاا عقولما,Bank's Name,كنبلا مسا,م
0,"Arwa Street, Crater, Aden - Yemen",نميلا -ندع-ىورأ عراش,_,Yemen Bank for Reconstruction & Development,يرمعتلاو ءاشنلال نيميلا كنبلا,1
1,"Arwa Street, Crater, Aden - Yemen",نميلا -ندع-ىورأ عراش,www.nbyemen.com,National Bank of Yemen,نيميلا يلهلأا كنبلا,2
2,"Al-Zubairy Street, Sana'a - Yemen",نميلا - ءاعنص - ييربزلا عراش,www.arabbank.com,Arab Bank PLC - Yemen,يبرعلا كنبلا,3
3,"Seera Street, Crater, Aden - Yemen",نميلا-ندع-ةيرص عراش,https://www.cacbankye.com,Cooperative & Agricultural Credit Bank (CAC Ba...,ندع - يعارزلاو ينواعتلا فيلستلا كنب,4
4,"Arwa Street, Crater, Aden - Yemen",نميلا -ندع-ىورأ عراش,https://www.ycb.bank/,Yemen Commercial Bank,نيميلا يراجتلا كنبلا,5
5,"Al-Zubairy Street, Sana'a - Yemen",نميلا - ءاعنص - ييربزلا عراش,http://www.iby-bank.com,Islamic Bank of Yemen for Finance and Investment,رامثتسلااو ليومتلل نيميلا يملاسلإا كنبلا,6
6,Jamal Street-Taiz-Yemen,نميلا-زعت-لاجم عراش,www.tadhamonbank.com,Tadhamon Bank,نماضتلا كنب,7
7,khormaksar-Aden-Yemen,نميلا-ندع-رسكمروخ,www.sababank.com,Saba Islamic Bank,يملاسلإا أبس كنب,8
8,"Arwa Street, Crater, Aden - Yemen",نميلا -ندع-ىورأ عراش,www.sbyb.net,Shamil Bank of Yemen & Bahrain,لماشلا نيرحبلا نميلا فرصم,9
9,"90th Street, Aden - Yemen",نميلا - ندع -ينعستلا عراش,www.alamalbank.com,Al-Amal Microfinance Bank,رغصلأا ليومتلل لملأا كنب,10


In [16]:
# Map columns flexibly
def pick_col(df, *needles):
    for c in df.columns:
        cl = str(c).lower()
        if any(n in cl for n in needles):
            return c
    return None

name_col = pick_col(df1, "bank's name", 'name')
addr_col = pick_col(df1, "bank's adress", 'adress')
site_col = pick_col(df1, 'ينوتركللاا عقولما', 'url', 'web')
print('cols:', name_col, addr_col, site_col)

for _, row in df1.iterrows():
    name = str(row[name_col]).replace('\n', ' ').strip() if name_col else ''
    if is_missing(name):
        continue
    address = str(row[addr_col]).replace('\n', ' ').strip() if addr_col else ''
    website = str(row[site_col]).replace('\n', ' ').strip() if site_col else ''
    if not website:
        address, website = split_address_website(address)

    sqldict['Name'].append(name)
    sqldict['Address_1'].append(address)
    sqldict['Website'].append(website if len(website) > 2 else '')
    sqldict['Cntry'].append('YE')
    sqldict['ListProcessDate'].append(processdate)
    sqldict['RegulationType'].append('Regulated')
    sqldict['RegCtry'].append(reg.split()[0])
    sqldict['RegCode'].append(reg.split()[1])
    sqldict['ListCode'].append(reg.split()[2])
    sqldict['ListName'].append(Typology[reg])
    

sqldict = bourange_same_length_array(sqldict)
clear_tempfolder(tempfolder)
print(f"Rows after list 1: {len(sqldict['ListProcessDate'])}")

cols: Bank's Name Bank's Adress ينوتركللاا عقولما
Rows after list 1: 26


In [17]:
df_sql = pd.DataFrame(sqldict)

In [18]:
df_sql

,bvdid,priority,ListLabel,Typology,EntryType,Name,InternalID_1,InternalID_1_type,InternalID_2,InternalID_2_type,...,LEI Code,BIC SWIFT Code,Name - Mother Company,Address_1 - Mother company,Address_2 - Mother company,City - Mother company,Zip - Mother company,Cntry - Mother company,Phone - Mother company,Check
0,,,,,,Yemen Bank for Reconstruction & Development,,,,,...,,,,,,,,,,
1,,,,,,National Bank of Yemen,,,,,...,,,,,,,,,,
2,,,,,,Arab Bank PLC - Yemen,,,,,...,,,,,,,,,,
3,,,,,,Cooperative & Agricultural Credit Bank (CAC Ba...,,,,,...,,,,,,,,,,
4,,,,,,Yemen Commercial Bank,,,,,...,,,,,,,,,,
5,,,,,,Islamic Bank of Yemen for Finance and Investment,,,,,...,,,,,,,,,,
6,,,,,,Tadhamon Bank,,,,,...,,,,,,,,,,
7,,,,,,Saba Islamic Bank,,,,,...,,,,,,,,,,
8,,,,,,Shamil Bank of Yemen & Bahrain,,,,,...,,,,,,,,,,
9,,,,,,Al-Amal Microfinance Bank,,,,,...,,,,,,,,,,


## List 2 — Licensed exchange companies (Arabic PDFs, translate to English)

In [9]:
reg = 'YE CBYE 2'
print(f'Working with list {reg}')

# pdfplumber returns Arabic cells in VISUAL (RTL-reversed) order.
# We reverse char order per cell to recover logical Arabic, then match headers
# against logical Arabic tokens. Available fields per inspection of the two PDFs:
#   PDF "شركات الصرافة" (companies, 8 cols): name, director, hq_city, address,
#                                            legal_entity, branches, renewal
#   PDF "منشآت الصرافة" (establishments, 7 cols): name, owner, governorate (city),
#                                                  address, renewal, legal_entity
# There are NO phone, license-number, or license-date columns in either PDF.

# Visual-order Arabic tokens (pdfplumber returns RTL text reversed). Headers
# may have wrap-orphan letters, so we match short, distinctive visual fragments.
FIELD_VISUAL_TOKENS = {
    'name':         ['ةكشلا مسا', 'ةأشنملا مسا'],
    'address':      ['ناونعلا'],
    'city':         ['ةظفاحملا', 'سيئرلا زكرملا', 'ةنيدملا'],
    'owner':        ['كلاملا مسا', 'يذيفنتلا ريدملا مسا', 'ريدملا مسا'],
    'legal_entity': ['نوناقلا نايكلا'],
    'renewal':      ['ديدجتلا', 'ديدج / ديدجت', 'ديدجت / ديدج'],
    'number':       ['مقرلا'],
}

def _to_logical(s):
    """Reverse pdfplumber's visual-order Arabic into logical reading order."""
    if not s:
        return ''
    # Cells often contain mid-word newlines from PDF wrapping; drop them.
    s = str(s).replace('\n', '').replace('\r', '')
    s = s[::-1]                          # visual -> logical (char reversal)
    s = re.sub(r'\s+', ' ', s).strip()
    return s

def _match_field(cell_visual):
    """Match visual-order Arabic header substring -> field name."""
    cv = re.sub(r'\s+', ' ', str(cell_visual or '').replace('\n', ' ')).strip()
    for field, toks in FIELD_VISUAL_TOKENS.items():
        if any(tok in cv for tok in toks):
            return field
    return None

def _build_col_map(header_row):
    """Map field name -> column index by scanning the header row (visual Arabic)."""
    col_map = {}
    for i, cell in enumerate(header_row):
        f = _match_field(cell)
        if f and f not in col_map:
            col_map[f] = i
    return col_map

def _is_header_row(row, col_map):
    """A repeated header row will match the same name token at the same position."""
    if 'name' not in col_map:
        return False
    idx = col_map['name']
    if idx >= len(row):
        return False
    cv = re.sub(r'\s+', ' ', str(row[idx] or '').replace('\n', ' ')).strip()
    return any(tok in cv for tok in FIELD_VISUAL_TOKENS['name'])

def extract_arabic_pdf(path):
    """Return a list of dicts with logical-Arabic field values."""
    out = []
    with pdfplumber.open(path) as pdf:
        col_map = None
        header_keys = ('name', 'address')  # minimum required to consider a row a header
        for page in pdf.pages:
            tbl = page.extract_table()
            if not tbl:
                continue
            start = 0
            if col_map is None:
                # Locate header row on this page
                for i, row in enumerate(tbl):
                    cmap = _build_col_map(row)
                    if all(k in cmap for k in header_keys):
                        col_map = cmap
                        start = i + 1
                        print(f'  [{os.path.basename(path)}] header @ page-row {i}: {col_map}')
                        break
                if col_map is None:
                    continue  # try next page
            for row in tbl[start:]:
                if not row or _is_header_row(row, col_map):
                    continue
                rec = {}
                for field, idx in col_map.items():
                    rec[field] = _to_logical(row[idx]) if idx < len(row) else ''
                if not rec.get('name'):
                    continue
                out.append(rec)
    return out

all_records = []
clear_tempfolder(tempfolder)
for url in PdfLinks[reg]:
    p = download_pdf(url, tempfolder)
    recs = extract_arabic_pdf(p)
    print(f'  {os.path.basename(p)}: {len(recs)} rows')
    all_records.extend(recs)

print(f'Total exchange entities collected: {len(all_records)}')
for s in all_records[:5]:
    print(s)

Working with list YE CBYE 2
[download] https://english.cby-ye.com/files/615d6aedbf960.pdf -> C:\Users\wuj1\OneDrive - Moody's\Desktop\Regulator\YE CBYE\tempfolder\615d6aedbf960.pdf (510266 bytes)
  [615d6aedbf960.pdf] header @ page-row 0: {'renewal': 0, 'legal_entity': 2, 'address': 3, 'city': 4, 'owner': 5, 'name': 6, 'number': 7}
  615d6aedbf960.pdf: 76 rows
[download] https://english.cby-ye.com/files/615d6b5e11ba1.pdf -> C:\Users\wuj1\OneDrive - Moody's\Desktop\Regulator\YE CBYE\tempfolder\615d6b5e11ba1.pdf (596299 bytes)
  [615d6b5e11ba1.pdf] header @ page-row 0: {'legal_entity': 0, 'renewal': 1, 'address': 2, 'city': 3, 'owner': 4, 'name': 5, 'number': 6}
  615d6b5e11ba1.pdf: 201 rows
Total exchange entities collected: 277
{'renewal': 'تجديد', 'legal_entity': 'شكةر', 'address': 'يشارع العواض / بجوار العروي للرصافة', 'city': 'ز تعز', 'owner': 'عبدالنارص محمد عبدالحق عبدالعزي', 'name': 'شكة النارص للرصافةر', 'number': '1'}
{'renewal': 'تجديد', 'legal_entity': 'شكةر', 'address': 'يال

In [10]:
# Translate unique Arabic strings ONCE (cached), then populate sqldict.
# Pre-batch unique values across name/address/city to minimize translator calls.
unique_ar = set()
for rec in all_records:
    for k in ('name', 'address', 'city', 'legal_entity'):
        v = rec.get(k, '')
        if v and not is_missing(v):
            unique_ar.add(v)

print(f'Unique Arabic strings to translate: {len(unique_ar)}')
for i, s in enumerate(sorted(unique_ar), 1):
    translate_text(s, src='ar')   # populates _translation_cache
    if i % 25 == 0:
        print(f'  translated {i}/{len(unique_ar)}')
print('Translation pass complete.')

for rec in all_records:
    name_ar = rec.get('name', '')
    addr_ar = rec.get('address', '')
    city_ar = rec.get('city', '')
    legal_ar = rec.get('legal_entity', '')
    renewal_ar = rec.get('renewal', '')

    name_en  = translate_text(name_ar, src='ar')
    addr_en  = translate_text(addr_ar, src='ar') if addr_ar else ''
    city_en  = translate_text(city_ar, src='ar') if city_ar else ''
    legal_en = translate_text(legal_ar, src='ar') if legal_ar else ''

    sqldict['Name'].append(name_ar)
    sqldict['Name - Mother Company'].append(name_en)
    sqldict['Address_1'].append(addr_ar)
    sqldict['Address_1 - Mother company'].append(addr_en)
    sqldict['City'].append(city_en)
    sqldict['City - Mother company'].append(city_ar)
    sqldict['License_Type'].append(legal_en)
    sqldict['Check'].append(renewal_ar)
    sqldict['Cntry'].append('Yemen')
    sqldict['ListProcessDate'].append(processdate)
    sqldict['RegulationType'].append('Regulated')
    sqldict['RegCtry'].append(reg.split()[0])
    sqldict['RegCode'].append(reg.split()[1])
    sqldict['ListCode'].append(reg.split()[2])
    sqldict['ListName'].append(Typology[reg])
    sqldict['ListLanguage'].append('Arabic')

sqldict = bourange_same_length_array(sqldict)
clear_tempfolder(tempfolder)
print(f"Rows after list 2: {len(sqldict['ListProcessDate'])}")

Unique Arabic strings to translate: 520
[translate] attempt 1/3 failed: 'coroutine' object has no attribute 'text'


C:\Users\wuj1\AppData\Local\Temp\2\ipykernel_52028\3797707911.py:42: RuntimeWarning: coroutine 'Translator.translate' was never awaited
  last_err = e


[translate] attempt 2/3 failed: 'coroutine' object has no attribute 'text'
[translate] attempt 3/3 failed: 'coroutine' object has no attribute 'text'
[translate] giving up, using original. last error: 'coroutine' object has no attribute 'text'
[translate] attempt 1/3 failed: 'coroutine' object has no attribute 'text'
[translate] attempt 2/3 failed: 'coroutine' object has no attribute 'text'
[translate] attempt 3/3 failed: 'coroutine' object has no attribute 'text'
[translate] giving up, using original. last error: 'coroutine' object has no attribute 'text'
[translate] attempt 1/3 failed: 'coroutine' object has no attribute 'text'
[translate] attempt 2/3 failed: 'coroutine' object has no attribute 'text'
[translate] attempt 3/3 failed: 'coroutine' object has no attribute 'text'
[translate] giving up, using original. last error: 'coroutine' object has no attribute 'text'
[translate] attempt 1/3 failed: 'coroutine' object has no attribute 'text'
[translate] attempt 2/3 failed: 'coroutine'

C:\Users\wuj1\AppData\Roaming\Python\Python312\site-packages\pygments\token.py:37: RuntimeWarning: coroutine 'Translator.translate' was never awaited
  new = _TokenType(self + (val,))


KeyboardInterrupt: 

In [ ]:
!pip install -q deep-translator


[notice] A new release of pip is available: 25.3 -> 26.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [ ]:
from deep_translator import GoogleTranslator

def _provider_translate(text: str, src: str, dest: str) -> str:
    return GoogleTranslator(source=src, target=dest).translate(text)

In [ ]:
for rec in all_records[:2]:
    name_ar = rec.get('name', '')
    addr_ar = rec.get('address', '')
    city_ar = rec.get('city', '')
    legal_ar = rec.get('legal_entity', '')
    renewal_ar = rec.get('renewal', '')

    print(f"Name: {name_ar} | Address: {addr_ar} | City: {city_ar} | Legal Entity: {legal_ar} | Renewal: {renewal_ar}")
    print(_provider_translate(name_ar, src='ar', dest='en'))



Name: شكة النارص للرصافةر | Address: يشارع العواض / بجوار العروي للرصافة | City: ز تعز | Legal Entity: شكةر | Renewal: تجديد


SSLError: HTTPSConnectionPool(host='translate.google.com', port=443): Max retries exceeded with url: /m?tl=en&sl=ar&q=%D8%B4%D9%83%D8%A9+%D8%A7%D9%84%D9%86%D8%A7%D8%B1%D8%B5+%D9%84%D9%84%D8%B1%D8%B5%D8%A7%D9%81%D8%A9%D8%B1 (Caused by SSLError(SSLCertVerificationError(1, '[SSL: CERTIFICATE_VERIFY_FAILED] certificate verify failed: self-signed certificate in certificate chain (_ssl.c:1000)')))

In [ ]:
#------------------------------------------------ Begin_writer ----------------------------------------
os.chdir(scriptfolder)
df = pd.DataFrame(sqldict)
df = df.drop_duplicates()
df.to_excel(filename, sheet_name='SQL Ready', index=False)
print(f'Saved -> {filename} (rows: {len(df)})')